# Database Table Preview (head 15)

This notebook previews table contents as pandas DataFrames, one database at a time and one section per step. It’s configured for a local MySQL instance using your `.env` variables.

In [ ]:
# Section 1: Configure Environment and Dependencies
# (Optional) Ensure required packages are available in the current env
# You can skip if these are already installed in your venv.
#
# import sys, subprocess
# pkgs = ["pandas", "SQLAlchemy", "pymysql", "python-dotenv"]
# for p in pkgs:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", p])

from dotenv import load_dotenv
load_dotenv()  # load .env into environment

In [ ]:
# If pandas is missing, install it here (run once)
try:
    import pandas as _pd  # noqa: F401
except ModuleNotFoundError:
    import sys, subprocess
    print('Installing pandas ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas>=2.2.3,<3'])
    import pandas as _pd  # retry
print('pandas ready')

In [ ]:
# Section 2: Import Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, inspect, text

ROW_LIMIT = 15

DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = int(os.getenv('DB_PORT', '3306'))
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
# Prefer private DB for exploration; fall back to DB_NAME only if private not set
DEFAULT_DB = os.getenv('DB_NAME_PRIVATE') or os.getenv('DB_NAME')

print({'host': DB_HOST, 'port': DB_PORT, 'default_db': DEFAULT_DB})

In [ ]:
# Section 3: Define Connection URL Template and Helpers
from urllib.parse import quote_plus

DB_URL_TEMPLATE = "mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4"


def make_engine(database: str):
    url = DB_URL_TEMPLATE.format(
        user=quote_plus(DB_USER or ''),
        password=quote_plus(DB_PASS or ''),
        host=DB_HOST,
        port=DB_PORT,
        database=quote_plus(database or ''),
    )
    return create_engine(url, pool_pre_ping=True)

In [ ]:
# Section 4: Select Databases (local-only)
# We only target the local DB from .env
databases = [DEFAULT_DB]
print({'databases': databases})

In [ ]:
# Section 5: Select CURRENT_DB (one at a time)
# Pick the target DB; default to DEFAULT_DB from .env (prefer DB_NAME_PRIVATE)
CURRENT_DB = DEFAULT_DB
assert CURRENT_DB, 'No DEFAULT_DB resolved from env (DB_NAME_PRIVATE/DB_NAME)'
assert CURRENT_DB in databases, f"{CURRENT_DB} not found on server"
print({'CURRENT_DB': CURRENT_DB, 'note': 'using private DB if provided'})

In [ ]:
# Section 6: Create Engine for Selected Database and test
engine = make_engine(CURRENT_DB)
with engine.connect() as conn:
    ok = conn.execute(text("SELECT 1")).scalar()
print({'connected': bool(ok)})

In [ ]:
# Section 7: Inspect Tables in CURRENT_DB
insp = inspect(engine)
# For MySQL, use the current database as schema
schema = CURRENT_DB
pairs = []
try:
    tables = insp.get_table_names(schema=schema)
except Exception:
    tables = []
for t in tables:
    pairs.append((schema, t))

print({'schema': schema})
print({'tables': pairs, 'count': len(pairs)})

In [ ]:
# Section 8: Preview CURRENT_TABLE (head 15)
CURRENT_SCHEMA = None  # For MySQL, schema can be None or the DB name
CURRENT_TABLE = 'youtube_videos_raw'  # change as needed

qualified = f"`{CURRENT_TABLE}`" if not CURRENT_SCHEMA else f"`{CURRENT_SCHEMA}`.`{CURRENT_TABLE}`"
query = text(f"SELECT * FROM {qualified} LIMIT :lim")

with engine.connect() as conn:
    df = pd.read_sql_query(query, conn, params={'lim': ROW_LIMIT})

df.head(ROW_LIMIT)

In [ ]:
# Section 9: Preview All Tables in CURRENT_DB (head 15 each)
from IPython.display import display

with engine.connect() as conn:
    for schema, table in pairs:
        qualified = f"`{table}`" if not schema else f"`{schema}`.`{table}`"
        print(f"\n=== {qualified} (head {ROW_LIMIT}) ===")
        try:
            q = text(f"SELECT * FROM {qualified} LIMIT :lim")
            df = pd.read_sql_query(q, conn, params={'lim': ROW_LIMIT})
            display(df.head(ROW_LIMIT))
        except Exception as e:
            # If a table disappears between inspect and query, skip gracefully
            print('SKIP:', table, type(e).__name__, str(e))

In [ ]:
# Section 10: Optional — Save Table Previews to CSV
SAVE_TO_CSV = False
OUTPUT_DIR = os.path.join(os.getcwd(), 'previews', CURRENT_DB)

if SAVE_TO_CSV:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with engine.connect() as conn:
        for schema, table in pairs:
            qualified = f"`{table}`" if not schema else f"`{schema}`.`{table}`"
            q = text(f"SELECT * FROM {qualified} LIMIT :lim")
            df = pd.read_sql_query(q, conn, params={'lim': ROW_LIMIT})
            name = f"{schema+'.' if schema else ''}{table}.head{ROW_LIMIT}.csv"
            df.to_csv(os.path.join(OUTPUT_DIR, name), index=False)
    print({'saved_to': OUTPUT_DIR})

In [ ]:
# Section 11: Cleanup: Dispose Engines
try:
    server_engine.dispose()
except Exception:
    pass
try:
    engine.dispose()
except Exception:
    pass
print('Disposed engines')

In [ ]:
# Section 8b: Preview specific local tables (head 15) — only if they exist
LOCAL_TABLES = [
    'youtube_comments',
    'youtube_metrics',
    'youtube_playlists_raw',
    'youtube_videos',
    'youtube_videos_raw',
]

from IPython.display import display

# Build a set of existing tables from Section 7
try:
    existing_tables = {t for _, t in pairs}
except Exception:
    existing_tables = set()

with engine.connect() as conn:
    for t in LOCAL_TABLES:
        if t not in existing_tables:
            print(f"\n=== `{CURRENT_DB}`.`{t}` (head {ROW_LIMIT}) ===")
            print('SKIP: table does not exist in current DB')
            continue
        qualified = f"`{CURRENT_DB}`.`{t}`"
        print(f"\n=== {qualified} (head {ROW_LIMIT}) ===")
        try:
            q = text(f"SELECT * FROM {qualified} LIMIT :lim")
            df = pd.read_sql_query(q, conn, params={'lim': ROW_LIMIT})
            display(df.head(ROW_LIMIT))
        except Exception as e:
            print('ERROR:', t, type(e).__name__, str(e))

In [ ]:
# Helpers: run query and clean artist/video names
import re


def get_df(sql_text, params=None):
    params = params or {}
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql_text), conn, params=params)


def clean_artist_name(name: str | None) -> str:
    if not name:
        return ''
    s = name.strip()
    # Remove leading @ from handles
    if s.startswith('@'):
        s = s[1:]
    # Drop YouTube ' - Topic' suffix
    s = re.sub(r"\s+-\s+Topic$", "", s, flags=re.IGNORECASE)
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s)
    return s


def clean_video_title(title: str | None) -> str:
    """Lightweight normalization of video/song titles.
    - Remove official/vertical video tags
    - Remove trailing parentheses/brackets with 'Official Video' etc
    - Collapse whitespace
    """
    if not title:
        return ''
    t = title.strip()
    # Common decorations
    t = re.sub(r"\b(official|audio|video|visualizer|lyric|lyrics|mv)\b", "", t, flags=re.IGNORECASE)
    # Remove content in [] or () if it looks like descriptors
    t = re.sub(r"\s*[\[(].{0,40}?(official|audio|video|visualizer|lyric|lyrics|mv).{0,40}?[\])]", "", t, flags=re.IGNORECASE)
    # Remove '(Official ...)' like chunks
    t = re.sub(r"\s*\((?:official|audio|video|visualizer|lyric|lyrics|mv)[^)]*\)", "", t, flags=re.IGNORECASE)
    # Collapse extra separators like ' -  - '
    t = re.sub(r"\s*-\s*-+\s*", " - ", t)
    # Normalize whitespace
    t = re.sub(r"\s+", " ", t)
    return t.strip()

In [ ]:
# Unified previews joined with channel/artist names and cleaned video names (head 15)
from IPython.display import display

# Set of existing tables
try:
    existing_tables = {t for _, t in pairs}
except Exception:
    existing_tables = set()

# youtube_videos: already has channel_title and title
if 'youtube_videos' in existing_tables:
    try:
        df_vids = get_df(
            f"SELECT video_id, title, channel_title, published_at, view_count, like_count, comment_count FROM `{CURRENT_DB}`.`youtube_videos` ORDER BY published_at DESC LIMIT :lim",
            {'lim': ROW_LIMIT},
        )
        df_vids['artist'] = df_vids['channel_title'].map(clean_artist_name)
        df_vids['video_clean'] = df_vids['title'].map(clean_video_title)
        print("youtube_videos → head")
        display(df_vids.head(ROW_LIMIT))
    except Exception as e:
        print('youtube_videos preview error:', type(e).__name__, str(e))
else:
    print('youtube_videos → SKIP (table not found)')

# youtube_videos_raw: extract artist and title from raw JSON if present
if 'youtube_videos_raw' in existing_tables:
    try:
        df_raw = get_df(
            f"""
            SELECT video_id,
                   JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.channelTitle')) AS channel_title,
                   JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.title')) AS title,
                   fetched_at
            FROM `{CURRENT_DB}`.`youtube_videos_raw`
            ORDER BY fetched_at DESC
            LIMIT :lim
            """,
            {'lim': ROW_LIMIT},
        )
        df_raw['artist'] = df_raw['channel_title'].map(clean_artist_name)
        df_raw['video_clean'] = df_raw['title'].map(clean_video_title)
        print("youtube_videos_raw → head")
        display(df_raw.head(ROW_LIMIT))
    except Exception as e:
        print('youtube_videos_raw preview error:', type(e).__name__, str(e))
else:
    print('youtube_videos_raw → SKIP (table not found)')

# youtube_metrics: join to videos_raw for channel title and title
if 'youtube_metrics' in existing_tables and 'youtube_videos_raw' in existing_tables:
    try:
        df_met = get_df(
            f"""
            SELECT m.video_id,
                   m.metrics_date,
                   m.view_count,
                   m.like_count,
                   m.comment_count,
                   JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelTitle')) AS channel_title,
                   JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.title')) AS title
            FROM `{CURRENT_DB}`.`youtube_metrics` m
            LEFT JOIN `{CURRENT_DB}`.`youtube_videos_raw` r ON r.video_id = m.video_id
            ORDER BY m.metrics_date DESC, m.view_count DESC
            LIMIT :lim
            """,
            {'lim': ROW_LIMIT},
        )
        df_met['artist'] = df_met['channel_title'].map(clean_artist_name)
        df_met['video_clean'] = df_met['title'].map(clean_video_title)
        print("youtube_metrics (joined) → head")
        display(df_met.head(ROW_LIMIT))
    except Exception as e:
        print('youtube_metrics preview error:', type(e).__name__, str(e))
else:
    print('youtube_metrics (joined) → SKIP (metrics/raw table missing)')

# youtube_comments: join to videos_raw for channel title and title
if 'youtube_comments' in existing_tables and 'youtube_videos_raw' in existing_tables:
    try:
        df_com = get_df(
            f"""
            SELECT c.id,
                   c.video_id,
                   c.author_name,
                   c.like_count,
                   c.published_at,
                   JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelTitle')) AS channel_title,
                   JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.title')) AS title,
                   c.comment_text
            FROM `{CURRENT_DB}`.`youtube_comments` c
            LEFT JOIN `{CURRENT_DB}`.`youtube_videos_raw` r ON r.video_id = c.video_id
            ORDER BY c.published_at DESC
            LIMIT :lim
            """,
            {'lim': ROW_LIMIT},
        )
        df_com['artist'] = df_com['channel_title'].map(clean_artist_name)
        df_com['video_clean'] = df_com['title'].map(clean_video_title)
        print("youtube_comments (joined) → head")
        display(df_com.head(ROW_LIMIT))
    except Exception as e:
        print('youtube_comments preview error:', type(e).__name__, str(e))
else:
    print('youtube_comments (joined) → SKIP (comments/raw table missing)')

# youtube_playlists_raw: extract channelTitle and title from raw_data if present
if 'youtube_playlists_raw' in existing_tables:
    try:
        df_pl = get_df(
            f"""
            SELECT playlist_id,
                   JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.channelTitle')) AS channel_title,
                   JSON_UNQUOTE(JSON_EXTRACT(raw_data, '$.snippet.title')) AS title,
                   fetched_at, processed, error
            FROM `{CURRENT_DB}`.`youtube_playlists_raw`
            ORDER BY fetched_at DESC
            LIMIT :lim
            """,
            {'lim': ROW_LIMIT},
        )
        df_pl['artist'] = df_pl['channel_title'].map(clean_artist_name)
        df_pl['video_clean'] = df_pl['title'].map(clean_video_title)
        print("youtube_playlists_raw → head")
        display(df_pl.head(ROW_LIMIT))
    except Exception as e:
        print('youtube_playlists_raw preview error:', type(e).__name__, str(e))
else:
    print('youtube_playlists_raw → SKIP (table not found)')

In [ ]:
# Section 12: Charts — Top songs and artists by views/likes
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

# Require metrics and raw tables
try:
    existing_tables = {t for _, t in pairs}
except Exception:
    existing_tables = set()

if 'youtube_metrics' in existing_tables and 'youtube_videos_raw' in existing_tables:
    base_sql = f"""
    SELECT
        m.video_id,
        MAX(m.view_count) AS max_views,
        MAX(m.like_count) AS max_likes,
        JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.channelTitle')) AS channel_title,
        JSON_UNQUOTE(JSON_EXTRACT(r.raw_data, '$.snippet.title')) AS title
    FROM `{CURRENT_DB}`.`youtube_metrics` m
    LEFT JOIN `{CURRENT_DB}`.`youtube_videos_raw` r ON r.video_id = m.video_id
    GROUP BY m.video_id, channel_title, title
    """

    df_base = get_df(base_sql)
    df_base['artist'] = df_base['channel_title'].map(clean_artist_name)
    df_base['video_clean'] = df_base['title'].map(clean_video_title)

    for col in ['max_views', 'max_likes']:
        if col in df_base.columns:
            df_base[col] = df_base[col].fillna(0)

    if plt is None:
        print('matplotlib not available; skipping charts. Install matplotlib to enable plotting.')
    else:
        # Top songs by views
        top_songs_views = df_base.sort_values('max_views', ascending=False).head(15)
        if not top_songs_views.empty:
            plt.figure(figsize=(10, 6))
            labels = top_songs_views['artist'].fillna('') + ' — ' + top_songs_views['video_clean'].fillna('')
            plt.barh(labels, top_songs_views['max_views'])
            plt.gca().invert_yaxis()
            plt.title('Top 15 songs by views')
            plt.xlabel('Views')
            plt.tight_layout()
            plt.show()
        else:
            print('No data for top songs by views')

        # Top songs by likes
        top_songs_likes = df_base.sort_values('max_likes', ascending=False).head(15)
        if not top_songs_likes.empty:
            plt.figure(figsize=(10, 6))
            labels = top_songs_likes['artist'].fillna('') + ' — ' + top_songs_likes['video_clean'].fillna('')
            plt.barh(labels, top_songs_likes['max_likes'])
            plt.gca().invert_yaxis()
            plt.title('Top 15 songs by likes')
            plt.xlabel('Likes')
            plt.tight_layout()
            plt.show()
        else:
            print('No data for top songs by likes')

        # Top artists by cumulative views
        artist_views = (
            df_base.groupby('artist', dropna=False)['max_views']
            .sum()
            .reset_index()
            .sort_values('max_views', ascending=False)
            .head(15)
        )
        if not artist_views.empty:
            plt.figure(figsize=(10, 6))
            plt.barh(artist_views['artist'].fillna(''), artist_views['max_views'])
            plt.gca().invert_yaxis()
            plt.title('Top 15 artists by total views')
            plt.xlabel('Views')
            plt.tight_layout()
            plt.show()
        else:
            print('No data for top artists by views')

        # Top artists by cumulative likes
        artist_likes = (
            df_base.groupby('artist', dropna=False)['max_likes']
            .sum()
            .reset_index()
            .sort_values('max_likes', ascending=False)
            .head(15)
        )
        if not artist_likes.empty:
            plt.figure(figsize=(10, 6))
            plt.barh(artist_likes['artist'].fillna(''), artist_likes['max_likes'])
            plt.gca().invert_yaxis()
            plt.title('Top 15 artists by total likes')
            plt.xlabel('Likes')
            plt.tight_layout()
            plt.show()
        else:
            print('No data for top artists by likes')
else:
    print('Charts skipped: required tables not present (need youtube_metrics and youtube_videos_raw).')

# Sentiment vs Plays (Views)

Compare average comment sentiment with maximum daily views per video.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
from web.db_guard import get_engine

load_dotenv(dotenv_path=Path().resolve() / '.env', override=True)
eng = get_engine('private', ro=True)

sql = '''
SELECT v.video_id, v.title, v.channel_title, v.isrc,
       s.avg_sentiment, m.max_views
FROM youtube_sentiment_summary s
JOIN (
  SELECT video_id, MAX(view_count) AS max_views
  FROM youtube_metrics
  GROUP BY video_id
) m USING (video_id)
JOIN youtube_videos v ON v.video_id = s.video_id
'''
df_sv = pd.read_sql(sql, eng)
print(f'Loaded {len(df_sv)} rows')


In [ ]:
# Scatter: sentiment vs log10(views)
if not df_sv.empty:
    dfp = df_sv.copy()
    dfp['views_log10'] = np.log10(dfp['max_views'].clip(lower=1))
    fig, ax = plt.subplots(figsize=(8,5))
    sc = ax.scatter(dfp['views_log10'], dfp['avg_sentiment'], c='tab:purple', alpha=0.35, edgecolors='none')
    ax.axhline(0.0, color='gray', lw=1, alpha=0.6)
    ax.set_xlabel('log10(Max Views)')
    ax.set_ylabel('Avg Sentiment (VADER)')
    ax.set_title('Sentiment vs Popularity (per video)')
    plt.show()
else:
    print('No sentiment/view data available')


In [ ]:
# Bars: Top 15 by views with sentiment overlay
if not df_sv.empty:
    top = df_sv.sort_values('max_views', ascending=False).head(15)
    labels = (top['channel_title'].fillna('') + ' — ' + top['title'].fillna('')).str.slice(0, 40)
    x = np.arange(len(top))
    fig, ax = plt.subplots(figsize=(9,6))
    bars = ax.bar(x, top['max_views'], color='tab:blue', alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot(x, top['avg_sentiment'], color='tab:red', marker='o', linewidth=1.5, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Max Views')
    ax2.set_ylabel('Avg Sentiment')
    ax.set_title('Top 15 Videos by Views — Sentiment Overlay')
    fig.tight_layout()
    plt.show()
else:
    print('No data for top videos by views')
